In [3]:
import numpy as np
import pandas as pd

# 1. CSV 데이터 불러오기
df_raw = pd.read_csv('knhanes_like_week4.csv')
df = df_raw.copy()

# 2. 2팀 노출 변수(과거흡연) 설정
df['exposure'] = (df['smk'] == 2).astype(int)

# 3. 피드백 반영: edu, income의 무응답 코드(9 또는 99)를 NaN으로 처리
df['edu'] = df['edu'].replace([9, 99], np.nan)
df['income'] = df['income'].replace([9, 99], np.nan)

w = df['wt'].to_numpy()
ST = df['kstrata'].to_numpy()
PS = df['psu'].to_numpy()

# 4. 설계 기반 분산 및 도메인 평균 계산 함수
def _var_psu(z, strata, psu):
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var

def svymean_domain(y, w, strata, psu, domain):
    y = np.asarray(y, float)
    d = np.asarray(domain, float) * (~np.isnan(y))
    y = np.nan_to_num(y) * d
    wd = np.asarray(w, float) * d
    th = np.sum(wd * y) / np.sum(wd)
    return th, np.sqrt(_var_psu(wd * (y - th) / np.sum(wd), strata, psu))

# 5. 팀 과제용 3개 열 도메인 정의
DOMS = {
    '전체': np.ones(len(df), bool),
    '과거흡연군 (노출)': df['exposure'] == 1,
    '비과거흡연군 (비노출)': df['exposure'] == 0
}

# 6. 행 생성 함수
def row_cont(label, var):
    """연속형: 가중 평균 (SE)"""
    r = {'특성': label, 'n': int(df[var].notna().sum())}
    for name, dom in DOMS.items():
        th, se = svymean_domain(df[var], w, ST, PS, dom)
        r[name] = f'{th:.1f} ({se:.2f})'
    return r

def row_cat(label, mask):
    """범주형: n (가중 %)"""
    mask = np.asarray(mask, float)
    r = {'특성': label, 'n': int(mask.sum())}
    for name, dom in DOMS.items():
        th, se = svymean_domain(mask, w, ST, PS, dom)
        r[name] = f'{th*100:.1f}'
    return r

# 7. Table 1 최종 생성 및 출력 (비만 행 제외, 신규 변수 5개 포함)
t1 = pd.DataFrame([
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df.age >= 65),
    row_cat('남성', df.sex == 1),
    row_cat('읍면 거주', df.urban == 2),
    row_cat('음주 (drink)', df.drink == 1),           # 음주 변수 추가
    row_cat('교육수준 (edu)', df.edu >= 3),           # 교육수준 변수 추가
    row_cont('소득 (income)', 'income'),              # 소득 변수 추가
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cont('공복혈당 (glucose)', 'glucose'),        # 공복혈당 변수 추가
    row_cont('총콜레스테롤 (chol)', 'chol'),           # 총콜레스테롤 변수 추가
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('고혈압', df.htn == 1),
    row_cat('규칙적 운동', df.exercise == 2),
])

t1

,특성,n,전체,과거흡연군 (노출),비과거흡연군 (비노출)
0,연령 (세),5367,49.1 (0.16),50.0 (0.53),49.0 (0.19)
1,65세 이상,1607,16.8,18.0,16.6
2,남성,2719,49.0,40.8,50.6
3,읍면 거주,2156,22.9,24.1,22.7
4,음주 (drink),1627,30.1,27.6,30.6
5,교육수준 (edu),3526,72.9,72.7,72.9
6,소득 (income),5041,2.5 (0.02),2.6 (0.04),2.5 (0.02)
7,BMI (kg/m²),5367,23.8 (0.05),24.0 (0.14),23.8 (0.06)
8,공복혈당 (glucose),5000,91.6 (0.25),92.2 (0.61),91.5 (0.26)
9,총콜레스테롤 (chol),4806,170.4 (0.49),170.2 (1.25),170.4 (0.53)


In [4]:
# 비가중 Table 1 수치 계산 근거 출력
import pandas as pd

# 1. 분석 대상 주요 변수 리스트 (노트북 실제 변수명 적용)
target_vars = {
    '65세 이상': df['age'] >= 65,
    '남성': df['sex'] == 1,
    '읍면 거주': df['urban'] == 2,
    '비만 (BMI≥25)': df['bmi'] >= 25,
    '고혈압': df['htn'] == 1,
    '규칙적 운동': df['exercise'] == 2
}

# 2. 비가중 빈도(n) 및 비가중 비율(%) 계산
unweighted_results = []
for name, condition in target_vars.items():
    n = condition.sum()
    total = condition.dropna().count()
    pct = (n / total) * 100
    unweighted_results.append({
        '특성': name,
        '비가중 n': int(n),
        '비가중 비율(%)': round(pct, 1)
    })

# 3. 데이터프레임 변환 및 출력
unweighted_tb = pd.DataFrame(unweighted_results)
print(f"전체 표본 수 (N): {len(df)}")
display(unweighted_tb)

전체 표본 수 (N): 5367


,특성,비가중 n,비가중 비율(%)
0,65세 이상,1607,29.9
1,남성,2719,50.7
2,읍면 거주,2156,40.2
3,비만 (BMI≥25),2035,37.9
4,고혈압,1652,30.8
5,규칙적 운동,2238,41.7
